# LLM Math Reasoning Experiment with OpenAI-Compatible API

This notebook runs math-reasoning prompt experiments against an OpenAI v1-compatible chat completions API.

**Setup:**
1. Add the dataset containing `problem.math.csv`, for example `jakenty1/vnmathproblems2026`.
2. Provide a `.env` file or environment variables with:
   - `OPENAI_API_KEY`
   - `OPENAI_BASE_URL` or `OPENAI_URL`
   - optional `OPENAI_MODEL`
   - optional `DELAYS` in seconds between API calls

The notebook writes result files with `.math` in the output filename.


In [ ]:
# No local model packages are required for the OpenAI-compatible API runner.


In [ ]:
# No apt packages are required for the OpenAI-compatible API runner.


In [1]:
import os
import time
import requests
import json
import re
import csv
from datetime import datetime
from pathlib import Path


def load_env_file(path):
    """Small .env loader to avoid an extra dependency."""
    path = Path(path)
    if not path.exists():
        return False
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)
    return True


def load_env():
    """Load .env from common local/Kaggle locations without printing secrets."""
    candidate_paths = [
        Path(".env"),
        Path("/kaggle/working/.env"),
        Path("/kaggle/input/.env"),
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidate_paths.extend(kaggle_input.glob("**/.env"))

    loaded = []
    for path in candidate_paths:
        try:
            if load_env_file(path):
                loaded.append(str(path))
        except Exception as exc:
            print(f"Could not load {path}: {exc}")
    print(f"Loaded .env files: {loaded if loaded else 'none found; using existing environment variables'}")


load_env()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL = (
    os.getenv("OPENAI_BASE_URL")
    or os.getenv("OPENAI_URL")
    or os.getenv("OPENAI_API_BASE")
    or "https://api.openai.com/v1"
)
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
DELAYS = float(os.getenv("DELAYS", "0"))

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is missing. Add it to .env or environment variables.")

print("OpenAI-compatible API configured.")
print(f"Base URL: {OPENAI_BASE_URL}")
print(f"Model: {OPENAI_MODEL}")
print(f"Delay between API calls: {DELAYS}s")


Loaded .env files: ['.env']
OpenAI-compatible API configured.
Base URL: https://generativelanguage.googleapis.com/v1beta
Model: gemini-3.1-flash-lite-preview
Delay between API calls: 0.0s


In [ ]:
# OpenAI-compatible API mode does not need a local server.


In [ ]:
# Available models depend on your API provider and `OPENAI_MODEL` setting.


## 2. Math Configuration and Data Loading


In [ ]:
def chat_completions_url(base_url: str) -> str:
    base_url = base_url.rstrip("/")
    if base_url.endswith("/chat/completions"):
        return base_url

    # Gemini's OpenAI-compatible endpoint is:
    # https://generativelanguage.googleapis.com/v1beta/openai/chat/completions
    if "generativelanguage.googleapis.com" in base_url:
        if base_url.endswith("/openai"):
            return f"{base_url}/chat/completions"
        if base_url.endswith("/v1beta") or base_url.endswith("/v1"):
            return f"{base_url}/openai/chat/completions"
        return f"{base_url}/chat/completions"

    if base_url.endswith("/v1"):
        return f"{base_url}/chat/completions"
    return f"{base_url}/v1/chat/completions"


def call_openai_v1(model, system_prompt, user_prompt):
    start = time.time()
    try:
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": user_prompt})

        payload = {
            "model": model,
            "messages": messages,
            "temperature": 0.2,
            "max_tokens": 2048,
        }
        headers = {
            "Authorization": f"Bearer {OPENAI_API_KEY}",
            "Content-Type": "application/json",
        }
        r = requests.post(
            chat_completions_url(OPENAI_BASE_URL),
            headers=headers,
            json=payload,
            timeout=300,
        )
        r.raise_for_status()
        data = r.json()
        usage = data.get("usage", {})
        response = data["choices"][0]["message"].get("content", "")
        return response, {
            "duration": time.time() - start,
            "prompt_tokens": usage.get("prompt_tokens", 0),
            "completion_tokens": usage.get("completion_tokens", 0),
            "success": True,
            "error": "",
            "raw_finish_reason": data["choices"][0].get("finish_reason", ""),
        }
    except Exception as e:
        return "", {
            "duration": time.time() - start,
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "success": False,
            "error": str(e),
            "raw_finish_reason": "",
        }


def normalize_response_text(response: str) -> str:
    """Keep only the likely final answer from a model response."""
    text = response.strip()
    text = re.sub(r"```(?:text|json|python)?\s*", "", text, flags=re.IGNORECASE)
    text = text.replace("```", "").strip()

    marker_patterns = [
        r"(?:final answer|answer|result)\s*(?:is|:)?\s*(.+)$",
        r"(?:therefore|so),?\s*(.+)$",
    ]
    for pattern in marker_patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE | re.DOTALL)
        if matches:
            text = matches[-1].strip()
            break

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines:
        text = lines[-1]
    return text.strip().strip(".")


def extract_numbers(text: str):
    """Extract numeric values from text, including decimals and percentages."""
    text = normalize_response_text(text)
    values = []
    for match in re.finditer(r"[-+]?\d+(?:\.\d+)?\s*%?", text):
        token = match.group(0).strip()
        is_percent = token.endswith("%")
        token = token.rstrip("%")
        try:
            value = float(token)
        except ValueError:
            continue
        if is_percent:
            value /= 100
        values.append(value)
    return values


def parse_expected_answer(raw_value: str):
    """Parse expected_answer from problem.math.csv."""
    try:
        return json.loads(raw_value)
    except Exception:
        try:
            return float(raw_value)
        except Exception:
            return raw_value


def almost_equal(actual, expected, tolerance=1e-6):
    return abs(float(actual) - float(expected)) <= tolerance


def evaluate_math_answer(response_text: str, expected_answer, tolerance=1e-6):
    numbers = extract_numbers(response_text)
    expected_values = expected_answer if isinstance(expected_answer, list) else [expected_answer]

    errors = []
    if len(numbers) < len(expected_values):
        errors.append(
            f"Expected {len(expected_values)} numeric value(s), extracted {len(numbers)}: {numbers}"
        )
        return {
            "passed": False,
            "values_passed": 0,
            "values_total": len(expected_values),
            "extracted_values": numbers,
            "errors": errors,
        }

    candidate_values = numbers[-len(expected_values):]
    passed_count = 0
    for actual, expected in zip(candidate_values, expected_values):
        if almost_equal(actual, expected, tolerance=tolerance):
            passed_count += 1
        else:
            errors.append(f"Expected {expected}, got {actual}")

    return {
        "passed": passed_count == len(expected_values),
        "values_passed": passed_count,
        "values_total": len(expected_values),
        "extracted_values": candidate_values,
        "errors": errors,
    }


def parse_csv_row(row):
    return {
        "task_id": f"{row['task_id']}_{row['category_id']}",
        "base_task_id": row["task_id"],
        "category_id": row["category_id"],
        "description": row["prompt"].strip(),
        "expected_answer": parse_expected_answer(row["expected_answer"]),
        "difficulty": row.get("difficulty", ""),
        "source_task_id": row.get("source_task_id", ""),
        "source_old_id": row.get("source_old_id", ""),
    }


def get_completed_ids(filename):
    if not os.path.exists(filename):
        return set()
    completed = set()
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            try:
                data = json.loads(line)
                completed.add(data.get("task_id"))
            except Exception:
                continue
    return completed


def append_result(filename, result):
    with open(filename, "a", encoding="utf-8") as f:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")


csv_path = "/kaggle/input/datasets/jakenty1/vnmathproblems2026/problem.math.csv"
if not os.path.exists(csv_path):
    csv_path = "/kaggle/input/vnmathproblems2026/problem.math.csv"
if not os.path.exists(csv_path):
    csv_path = "/kaggle/working/problem.math.csv"

print(f"Using math problem CSV: {csv_path}")


In [ ]:
model_choice = OPENAI_MODEL
start_problem = 1
end_problem = 50
system_prompt = "Return only the final math result. Do not write code."

safe_model_name = re.sub(r"[^A-Za-z0-9_.-]+", "-", model_choice).strip("-")
output_filename = f"/kaggle/working/{safe_model_name}_{start_problem}_{end_problem}.math.jsonl"
completed_ids = get_completed_ids(output_filename)

problems_to_test = []
with open(csv_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        try:
            numeric_source_id = int(row.get("source_task_id") or row["task_id"].replace("math_", ""))
            if start_problem <= numeric_source_id <= end_problem:
                prob = parse_csv_row(row)
                if prob["task_id"] not in completed_ids:
                    problems_to_test.append(prob)
        except Exception:
            continue

print(f"Loaded {len(problems_to_test)} remaining math tasks for range {start_problem}-{end_problem}.")

for index, prob in enumerate(problems_to_test):
    if index > 0 and DELAYS > 0:
        time.sleep(DELAYS)
    print(f"Testing {prob['task_id']}...", end=" ", flush=True)
    response_text, meta = call_openai_v1(model_choice, system_prompt, prob["description"])
    if not meta["success"]:
        print(f"Error: {meta['error']}")
        result = {
            "timestamp": datetime.now().isoformat(),
            "model": model_choice,
            "task_id": prob["task_id"],
            "base_task_id": prob["base_task_id"],
            "category_id": prob["category_id"],
            "difficulty": prob["difficulty"],
            "source_task_id": prob["source_task_id"],
            "source_old_id": prob["source_old_id"],
            "passed": False,
            "values_passed": 0,
            "values_total": 1 if not isinstance(prob["expected_answer"], list) else len(prob["expected_answer"]),
            "expected_answer": prob["expected_answer"],
            "extracted_values": [],
            "errors": [meta["error"]],
            "duration_seconds": meta["duration"],
            "prompt_tokens": meta["prompt_tokens"],
            "completion_tokens": meta["completion_tokens"],
            "finish_reason": meta.get("raw_finish_reason", ""),
            "response_text": response_text,
        }
        append_result(output_filename, result)
        continue

    eval_res = evaluate_math_answer(response_text, prob["expected_answer"])
    print(f"{'PASS' if eval_res['passed'] else 'FAIL'} ({eval_res['values_passed']}/{eval_res['values_total']})")

    result = {
        "timestamp": datetime.now().isoformat(),
        "model": model_choice,
        "task_id": prob["task_id"],
        "base_task_id": prob["base_task_id"],
        "category_id": prob["category_id"],
        "difficulty": prob["difficulty"],
        "source_task_id": prob["source_task_id"],
        "source_old_id": prob["source_old_id"],
        "passed": eval_res["passed"],
        "values_passed": eval_res["values_passed"],
        "values_total": eval_res["values_total"],
        "expected_answer": prob["expected_answer"],
        "extracted_values": eval_res["extracted_values"],
        "errors": eval_res["errors"],
        "duration_seconds": meta["duration"],
        "prompt_tokens": meta["prompt_tokens"],
        "completion_tokens": meta["completion_tokens"],
        "finish_reason": meta.get("raw_finish_reason", ""),
        "response_text": response_text,
    }
    append_result(output_filename, result)

print(f"Results saved to {output_filename}")


## 4. Run Additional Ranges
Edit `start_problem`, `end_problem`, or `.env` `OPENAI_MODEL`, then rerun the final cell to process new math tasks. Output files use `.math.jsonl`.
